# Marathi WSD comparison: SenWiCh WiC on Kaggle

This notebook downloads the Marathi split files directly from the latest SenWiCh Zenodo record and fine-tunes MuRIL. It needs **Internet on** and a **GPU accelerator** in Kaggle. You do not need to upload the project ZIP or dataset.

SenWiCh is a research dataset with Marathi sense-annotated sentences and an official Word-in-Context (WiC) format. The model learns whether two contexts use a target lemma in the same sense. This is a related but different task from this project’s sentence–gloss compatibility model, so report these results as a separate benchmark; do not compare its accuracy directly with the project’s WSD top-1 score.

The notebook uses the release’s train/dev/test splits, selects the checkpoint on dev macro-F1, and evaluates the test set once. It saves the best model, metrics, and test predictions under `/kaggle/working/senwich_marathi_wic`.

Dataset: [SenWiCh on Zenodo](https://zenodo.org/records/18803861), DOI `10.5281/zenodo.18803861`. Please cite Goworek et al., SIGTYP 2025, and check the record’s license before redistributing the data or model.

In [ ]:
# Kaggle Settings: turn Internet on and select GPU before running all cells.
import os, sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Notebook Settings, restart the session, and run all cells again.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Install only the model/data libraries; keep Kaggle's preinstalled CUDA-enabled PyTorch.
%pip -q install "transformers>=4.44,<5" "accelerate>=0.33" "sentencepiece>=0.2" "peft>=0.13,<1" "scikit-learn>=1.4,<2" requests

# Avoid an optional incompatible torchao dispatcher in some Kaggle images.
import subprocess
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)


In [ ]:
from pathlib import Path
import hashlib, json, re, zipfile, requests

WORK = Path("/kaggle/working/senwich_marathi_wic")
WORK.mkdir(parents=True, exist_ok=True)
RECORD_ID = "18803861"  # latest Zenodo release found on 2026-10-13
API_URL = f"https://zenodo.org/api/records/{RECORD_ID}"

response = requests.get(API_URL, timeout=60)
response.raise_for_status()
record = response.json()
metadata = record.get("metadata", {})
print("Dataset:", metadata.get("title", "SenWiCh"))
print("Record DOI:", metadata.get("doi", "10.5281/zenodo." + RECORD_ID))
print("License metadata:", metadata.get("license") or "Not specified in API metadata; check the Zenodo record before redistribution.")

files = record.get("files", [])
archive_meta = next((f for f in files if f.get("key", "").lower().endswith("formatted_wic_files.zip")), None)
if archive_meta is None:
    raise RuntimeError("Could not find formatted_wic_files.zip in Zenodo record files. Check the record page for a newer archive.")
archive_path = WORK / archive_meta["key"]
if not archive_path.exists() or archive_path.stat().st_size != archive_meta.get("size"):
    url = archive_meta.get("links", {}).get("self")
    if not url:
        url = f"https://zenodo.org/records/{RECORD_ID}/files/{archive_meta['key']}?download=1"
    with requests.get(url, stream=True, timeout=(30, 180)) as r:
        r.raise_for_status()
        with archive_path.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

expected = archive_meta.get("checksum", "")
if expected.startswith("md5:"):
    digest = hashlib.md5(archive_path.read_bytes()).hexdigest()
    if digest.lower() != expected.split(":", 1)[1].lower():
        raise RuntimeError("The downloaded archive failed Zenodo's MD5 check; delete it and run this cell again.")
print(f"Downloaded {archive_path.name}: {archive_path.stat().st_size / 1024**2:.1f} MiB")


In [ ]:
# Find the official Marathi train/dev/test WiC files inside the archive.
from pathlib import PurePosixPath

with zipfile.ZipFile(archive_path) as zf:
    members = [n for n in zf.namelist() if not n.endswith("/")]
    marathi_files = [n for n in members if "marathi" in PurePosixPath(n).name.lower()]
    print("Marathi files:", marathi_files)
    split_members = {}
    for split, patterns in {
        "train": ("train",), "dev": ("dev", "valid", "validation"), "test": ("test",)
    }.items():
        hits = [n for n in marathi_files if any(re.search(rf"(^|[^a-z]){p}([^a-z]|$)", PurePosixPath(n).stem.lower()) for p in patterns)]
        if not hits:
            hits = [n for n in marathi_files if any(p in PurePosixPath(n).stem.lower() for p in patterns)]
        if len(hits) != 1:
            raise RuntimeError(f"Expected one Marathi {split} file, found {hits}. Inspect the printed file list and adjust this notebook.")
        split_members[split] = (hits[0], zf.read(hits[0]))

print("Selected splits:", {k: v[0] for k, v in split_members.items()})


In [ ]:
# Parse JSON, JSONL, or tabular WiC files and normalize common WiC column names.
import csv, io

ALIASES = {
    "lemma": ["lemma", "word", "target", "target_word", "targetlemma"],
    "pos": ["pos", "partofspeech", "part_of_speech"],
    "sentence1": ["sentence1", "sentence_1", "sent1", "context1", "context_1", "text1", "sentencea"],
    "sentence2": ["sentence2", "sentence_2", "sent2", "context2", "context_2", "text2", "sentenceb"],
    "start1": ["start1", "start_1", "startchar1", "startindex1"],
    "end1": ["end1", "end_1", "endchar1", "endindex1"],
    "start2": ["start2", "start_2", "startchar2", "startindex2"],
    "end2": ["end2", "end_2", "endchar2", "endindex2"],
    "label": ["label", "gold_label", "same_sense", "issamesense", "target", "gold"],
}
ALIAS_TO_FIELD = {re.sub(r"[^a-z0-9]", "", alias.lower()): field for field, aliases in ALIASES.items() for alias in aliases}

def records_from_object(obj):
    found = []
    if isinstance(obj, list):
        for item in obj:
            found.extend(records_from_object(item))
    elif isinstance(obj, dict):
        keys = {re.sub(r"[^a-z0-9]", "", str(k).lower()) for k in obj}
        def has_field(field):
            return any(re.sub(r"[^a-z0-9]", "", alias.lower()) in keys for alias in ALIASES[field])
        if all(has_field(field) for field in ("lemma", "sentence1", "sentence2", "label")):
            found.append(obj)
        else:
            for value in obj.values():
                if isinstance(value, (list, dict)):
                    found.extend(records_from_object(value))
    return found

def parse_split(name, payload):
    text = payload.decode("utf-8-sig").strip()
    try:
        obj = json.loads(text)
        rows = records_from_object(obj)
        if rows:
            return rows
    except json.JSONDecodeError:
        pass
    rows = []
    for line in text.splitlines():
        if not line.strip():
            continue
        try:
            rows.extend(records_from_object(json.loads(line)))
        except json.JSONDecodeError:
            rows = []
            break
    if rows:
        return rows
    sample = text[:1000]
    delimiter = "\t" if "\t" in sample.splitlines()[0] else ","
    table = list(csv.DictReader(io.StringIO(text), delimiter=delimiter))
    if table and any(re.sub(r"[^a-z0-9]", "", k.lower()) in ALIAS_TO_FIELD for k in table[0]):
        return table
    raise ValueError(f"Could not parse {name} as WiC JSON/JSONL/table. First 1000 characters:\n{sample}")

def canonicalize(raw):
    row = {re.sub(r"[^a-z0-9]", "", str(k).lower()): v for k, v in raw.items()}
    out = {}
    for field, aliases in ALIASES.items():
        for alias in aliases:
            key = re.sub(r"[^a-z0-9]", "", alias.lower())
            if key in row and row[key] is not None:
                out[field] = row[key]
                break
    required = ("lemma", "sentence1", "sentence2", "label")
    if any(field not in out for field in required):
        raise ValueError(f"Missing required WiC fields {required}; available keys: {list(raw)}")
    label = str(out["label"]).strip().lower()
    if label in {"1", "true", "t", "yes", "same", "same_sense", "same sense"}:
        out["label"] = 1
    elif label in {"0", "false", "f", "no", "different", "different_sense", "different sense"}:
        out["label"] = 0
    else:
        try:
            out["label"] = int(float(label))
        except ValueError as exc:
            raise ValueError(f"Unrecognized WiC label {out['label']!r}") from exc
    if out["label"] not in (0, 1):
        raise ValueError(f"WiC labels must be 0/1; found {out['label']!r}")
    for key in ("lemma", "sentence1", "sentence2"):
        out[key] = str(out[key]).strip()
    out["pos"] = str(out.get("pos", ""))
    return out

splits = {}
for split, (filename, payload) in split_members.items():
    rows = [canonicalize(r) for r in parse_split(filename, payload)]
    rows = [r for r in rows if r["lemma"] and r["sentence1"] and r["sentence2"]]
    if not rows:
        raise RuntimeError(f"The Marathi {split} split is empty after parsing.")
    splits[split] = rows
    labels = [r["label"] for r in rows]
    print(f"{split}: {len(rows):,} pairs; positive={sum(labels):,}; negative={len(labels)-sum(labels):,}")
print("Example:", {k: splits["train"][0].get(k) for k in ("lemma", "pos", "sentence1", "sentence2", "label")})


In [ ]:
# Prepare target-marked sentence pairs and train MuRIL on Kaggle's GPU.
import math, time, random, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

SEED = 42
MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUM = 2
EPOCHS = 3
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

def mark_target(sentence, lemma, start=None, end=None):
    # Use character spans when valid; otherwise leave the context intact and provide the lemma explicitly.
    try:
        start, end = int(start), int(end)
        if 0 <= start < end <= len(sentence):
            return sentence[:start] + " [TGT] " + sentence[start:end] + " [/TGT] " + sentence[end:]
    except (TypeError, ValueError):
        pass
    return sentence

def make_text(row, index):
    context = row[f"sentence{index}"]
    context = mark_target(context, row["lemma"], row.get(f"start{index}"), row.get(f"end{index}"))
    pos = f" POS: {row['pos']}." if row.get("pos") else ""
    return f"Target lemma: {row['lemma']}.{pos} Context: {context}"

class WiCDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        return make_text(row, 1), make_text(row, 2), int(row["label"])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                  target_modules=["query", "value"], modules_to_save=["classifier"])
model = get_peft_model(base_model, lora).to(DEVICE)
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.print_trainable_parameters()

def collate(batch):
    left, right, labels = zip(*batch)
    encoded = tokenizer(list(left), list(right), truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")
    encoded["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoded

loaders = {
    name: DataLoader(WiCDataset(rows), batch_size=BATCH_SIZE, shuffle=(name == "train"), collate_fn=collate,
                     num_workers=2, pin_memory=True)
    for name, rows in splits.items()
}
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
steps_per_epoch = math.ceil(len(loaders["train"]) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=True)

@torch.no_grad()
def collect_predictions(loader):
    model.eval(); gold, probs = [], []
    for batch in loader:
        labels = batch.pop("labels").numpy().tolist()
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(**batch).logits
        probs.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
        gold.extend(labels)
    return np.asarray(gold), np.asarray(probs)

def metric_dict(gold, probs):
    pred = (probs >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(gold, pred)),
        "macro_f1": float(f1_score(gold, pred, average="macro", zero_division=0)),
        "positive_f1": float(f1_score(gold, pred, pos_label=1, zero_division=0)),
        "precision": float(precision_score(gold, pred, zero_division=0)),
        "recall": float(recall_score(gold, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(gold, probs)) if len(set(gold)) == 2 else None,
        "confusion_matrix": confusion_matrix(gold, pred, labels=[0, 1]).tolist(),
    }

BEST_DIR = WORK / "best_model"
if BEST_DIR.exists():
    import shutil; shutil.rmtree(BEST_DIR)
best_dev_f1, history = -1.0, []
for epoch in range(EPOCHS):
    start_time = time.time(); model.train(); optimizer.zero_grad(set_to_none=True); losses=[]; pending=0
    for step, batch in enumerate(loaders["train"]):
        pending += 1
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = model(**batch).loss
        scaler.scale(loss / GRAD_ACCUM).backward(); losses.append(float(loss.detach().cpu()))
        if pending == GRAD_ACCUM or (step + 1) == len(loaders["train"]):
            scaler.unscale_(optimizer)
            if pending < GRAD_ACCUM:
                for param in model.parameters():
                    if param.grad is not None: param.grad.mul_(GRAD_ACCUM / pending)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step(); pending = 0
        every = max(1, len(loaders["train"]) // 10)
        if (step + 1) % every == 0 or (step + 1) == len(loaders["train"]):
            print(f"Epoch {epoch+1}/{EPOCHS} | batch {step+1}/{len(loaders['train'])} | loss {np.mean(losses[-every:]):.4f}", flush=True)
    dev_gold, dev_probs = collect_predictions(loaders["dev"])
    dev_metrics = metric_dict(dev_gold, dev_probs)
    history.append({"epoch": epoch + 1, "train_loss": float(np.mean(losses)), "dev": dev_metrics})
    print(f"Epoch {epoch+1}: train_loss={np.mean(losses):.4f}, dev_accuracy={dev_metrics['accuracy']:.4f}, dev_macro_f1={dev_metrics['macro_f1']:.4f}, minutes={(time.time()-start_time)/60:.1f}", flush=True)
    if dev_metrics["macro_f1"] > best_dev_f1:
        best_dev_f1 = dev_metrics["macro_f1"]
        model.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
        print("Saved best dev checkpoint.", flush=True)

# Load best-dev checkpoint, then evaluate the held-out test split once.
del model
torch.cuda.empty_cache()
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = PeftModel.from_pretrained(base_model, BEST_DIR).to(DEVICE)
model.eval()
test_gold, test_probs = collect_predictions(loaders["test"])
test_metrics = metric_dict(test_gold, test_probs)


In [ ]:
# Save reproducible metrics and row-level predictions as Kaggle notebook outputs.
report = {
    "dataset": "SenWiCh Marathi WiC",
    "dataset_doi": metadata.get("doi", "10.5281/zenodo." + RECORD_ID),
    "dataset_record_id": RECORD_ID,
    "dataset_version": metadata.get("version"),
    "model": MODEL_NAME,
    "seed": SEED,
    "split_source": "official SenWiCh Marathi train/dev/test files",
    "split_rows": {name: len(rows) for name, rows in splits.items()},
    "best_dev_macro_f1": best_dev_f1,
    "training_history": history,
    "test_metrics": test_metrics,
}
(WORK / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
test_df = pd.DataFrame(splits["test"]).copy()
test_df["probability_same_sense"] = test_probs
test_df["predicted_label"] = (test_probs >= 0.5).astype(int)
test_df.to_csv(WORK / "test_predictions.csv", index=False, encoding="utf-8-sig")
print(json.dumps({"output_dir": str(WORK), "best_dev_macro_f1": best_dev_f1, "test_metrics": test_metrics}, indent=2))
print("Files:", sorted(p.name for p in WORK.iterdir()))
